In [1]:
from pathlib import Path
import gcamreader
import os
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

In [2]:
def to_Mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100-yr GWP
GWP_AR5 = {
    'CO2':    1,
    'CH4': 28,  # Methane
    'CH4_AGR': 28,  # Methane from Agriculture
    'CH4_AWB': 28,  # Methane from Agricultural Waste Burning
    'N2O': 265,  # Nitrous Oxide
    'N2O_AGR': 265,  # Nitrous Oxide from Agriculture
    'N2O_AWB': 265,  # Nitrous Oxide from Agricultural Waste Burning
    'HFC125': 3500,
    'HFC134a':1430,
    'HFC143a':4470,
    'HFC23':  14800,
    'HFC32':  675,
    'HFC43':  1500,
    'HFC227ea':3220,
    'HFC236fa':9810,
    'SF6':    23500,
    'C2F6':   12200,
    'CF4':    6630,
}

In [3]:
proj_path = Path("/data/project/tae/gcam-core")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [4]:
dbpath = "../output/"  # relative to current working directory
dbfile = "database_basexdb_korea_2035_20250721_7"
# dbfile = "database_basexdb_test"
conn = gcamreader.LocalDBConn(dbpath, dbfile)
queries = gcamreader.parse_batch_query(os.path.join('..', 'output', 'queries','Main_queries.xml'))

Database scenarios: Current-Policy, Current-Policy, Enhanced-Ambition


In [5]:
scenarios = list(conn.listScenariosInDB()['name'])
scenarios

['Current-Policy', 'Current-Policy', 'Enhanced-Ambition']

In [6]:
scenarios.reverse()

In [7]:
scenarios=['Current-Policy', 'Enhanced-Ambition']

In [8]:
for i, q in enumerate(queries):
    print(i, q.title)

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [363]:
i = 122
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea', 'USA'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

iron and steel prices


,Units,scenario,region,sector,Year,value
0,1975$/kg,Current-Policy,South Korea,iron and steel,1975,0.143380
1,1975$/kg,Current-Policy,South Korea,iron and steel,1990,0.131144
2,1975$/kg,Current-Policy,South Korea,iron and steel,2005,0.140740
3,1975$/kg,Current-Policy,South Korea,iron and steel,2010,0.139923
4,1975$/kg,Current-Policy,South Korea,iron and steel,2015,0.143446
...,...,...,...,...,...,...
83,1975$/kg,Enhanced-Ambition,USA,iron and steel,2080,0.000000
84,1975$/kg,Enhanced-Ambition,USA,iron and steel,2085,0.000000
85,1975$/kg,Enhanced-Ambition,USA,iron and steel,2090,0.000000
86,1975$/kg,Enhanced-Ambition,USA,iron and steel,2095,0.000000


In [364]:
df[(df['Year'].isin([2030, 2035]))]

,Units,scenario,region,sector,Year,value
7,1975$/kg,Current-Policy,South Korea,iron and steel,2030,0.165681
8,1975$/kg,Current-Policy,South Korea,iron and steel,2035,0.173376
29,1975$/kg,Current-Policy,USA,iron and steel,2030,0.177754
30,1975$/kg,Current-Policy,USA,iron and steel,2035,0.179851
51,1975$/kg,Enhanced-Ambition,South Korea,iron and steel,2030,0.189463
52,1975$/kg,Enhanced-Ambition,South Korea,iron and steel,2035,0.204327
73,1975$/kg,Enhanced-Ambition,USA,iron and steel,2030,0.177760
74,1975$/kg,Enhanced-Ambition,USA,iron and steel,2035,0.179858


In [365]:
i = 120
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

iron and steel production by tech and vintage


,Units,scenario,region,sector,subsector,technology,output,Year,value
0,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR CCS,year=2025",iron and steel,2025,0.464729
1,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR CCS,year=2025",iron and steel,2030,0.379862
2,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR CCS,year=2025",iron and steel,2035,0.232317
3,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR CCS,year=2030",iron and steel,2030,1.904150
4,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR CCS,year=2030",iron and steel,2035,1.556470
...,...,...,...,...,...,...,...,...,...
154,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2025",iron and steel,2030,6.387700
155,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2025",iron and steel,2035,3.906470
156,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2030",iron and steel,2030,12.176300
157,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2030",iron and steel,2035,9.954660


In [366]:
df[(df['Year']==2035)]

,Units,scenario,region,sector,subsector,technology,output,Year,value
2,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR CCS,year=2025",iron and steel,2035,0.232317
4,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR CCS,year=2030",iron and steel,2035,1.556470
5,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR CCS,year=2035",iron and steel,2035,3.296990
8,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR with hydrogen,year=2025",iron and steel,2035,0.242591
10,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR with hydrogen,year=2030",iron and steel,2035,0.881550
11,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR with hydrogen,year=2035",iron and steel,2035,1.204670
23,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR,year=2020",iron and steel,2035,1.556860
26,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR,year=2025",iron and steel,2035,6.613990
28,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR,year=2030",iron and steel,2035,14.991600
29,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR,year=2035",iron and steel,2035,10.840000


In [367]:
df[(df['Year'] == 2030) & (df['region'] == 'South Korea')]

,Units,scenario,region,sector,subsector,technology,output,Year,value
1,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR CCS,year=2025",iron and steel,2030,0.379862
3,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR CCS,year=2030",iron and steel,2030,1.904150
7,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR with hydrogen,year=2025",iron and steel,2030,0.396689
9,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR with hydrogen,year=2030",iron and steel,2030,1.078380
19,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR,year=2015",iron and steel,2030,8.843820
22,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR,year=2020",iron and steel,2030,4.267100
25,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR,year=2025",iron and steel,2030,10.814900
27,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR,year=2030",iron and steel,2030,18.336700
31,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"Biomass-based,year=2025",iron and steel,2030,0.499208
33,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"Biomass-based,year=2030",iron and steel,2030,1.280430


In [368]:
df[(df[''])]

KeyError: ''

In [ ]:
i = 120
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

iron and steel production by tech and vintage


,Units,scenario,region,sector,subsector,technology,output,Year,value
0,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR,year=1975",iron and steel,1975,0.665514
1,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR,year=1990",iron and steel,1990,13.226000
2,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR,year=2005",iron and steel,2005,26.767200
3,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR,year=2010",iron and steel,2010,34.108100
4,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,"BLASTFUR,year=2015",iron and steel,2015,48.479100
...,...,...,...,...,...,...,...,...,...
118,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2025",iron and steel,2030,6.417700
119,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2025",iron and steel,2035,3.924800
120,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2030",iron and steel,2030,12.700000
121,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2030",iron and steel,2035,10.382800


In [ ]:
df[(df['subsector'] == 'EAF with scrap') & (df['Year'] == 2035)]

,Units,scenario,region,sector,subsector,technology,output,Year,value
56,Mt,Current-Policy,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2020",iron and steel,2035,0.774412
59,Mt,Current-Policy,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2025",iron and steel,2035,3.944840
61,Mt,Current-Policy,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2030",iron and steel,2035,8.175470
62,Mt,Current-Policy,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2035",iron and steel,2035,11.781400
116,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2020",iron and steel,2035,0.774489
119,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2025",iron and steel,2035,3.924800
121,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2030",iron and steel,2035,10.382800
122,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2035",iron and steel,2035,22.117200


In [ ]:
i = 223
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df.head()

regional biomass consumption


,Units,scenario,region,sector,input,Year,value
0,EJ,Current-Policy,South Korea,regional biomass,total biomass,1975,0.039972
1,EJ,Current-Policy,South Korea,regional biomass,total biomass,1990,0.046407
2,EJ,Current-Policy,South Korea,regional biomass,total biomass,2005,0.109719
3,EJ,Current-Policy,South Korea,regional biomass,total biomass,2010,0.152805
4,EJ,Current-Policy,South Korea,regional biomass,total biomass,2015,0.229262


In [ ]:
df['input'].unique()

array(['total biomass'], dtype=object)

In [ ]:
df[(df['setor'])]

KeyError: 'setor'

In [ ]:
i = 124
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df.head()

regional iron and steel sources


,Units,scenario,region,sector,subsector,input,Year,value
0,Mt,Current-Policy,South Korea,regional iron and steel,domestic iron and steel,iron and steel,1975,1.063
1,Mt,Current-Policy,South Korea,regional iron and steel,domestic iron and steel,iron and steel,1990,15.553
2,Mt,Current-Policy,South Korea,regional iron and steel,domestic iron and steel,iron and steel,2005,31.696
3,Mt,Current-Policy,South Korea,regional iron and steel,domestic iron and steel,iron and steel,2010,34.286
4,Mt,Current-Policy,South Korea,regional iron and steel,domestic iron and steel,iron and steel,2015,38.497


In [ ]:
df[(df['Year'] >= 2030)]

,Units,scenario,region,sector,subsector,input,Year,value
7,Mt,Current-Policy,South Korea,regional iron and steel,domestic iron and steel,iron and steel,2030,40.0969
8,Mt,Current-Policy,South Korea,regional iron and steel,domestic iron and steel,iron and steel,2035,39.9669
17,Mt,Current-Policy,South Korea,regional iron and steel,imported iron and steel,traded iron and steel,2030,16.1407
18,Mt,Current-Policy,South Korea,regional iron and steel,imported iron and steel,traded iron and steel,2035,16.3652
26,Mt,Enhanced-Ambition,South Korea,regional iron and steel,domestic iron and steel,iron and steel,2030,27.6093
27,Mt,Enhanced-Ambition,South Korea,regional iron and steel,domestic iron and steel,iron and steel,2035,26.6431
36,Mt,Enhanced-Ambition,South Korea,regional iron and steel,imported iron and steel,traded iron and steel,2030,27.4903
37,Mt,Enhanced-Ambition,South Korea,regional iron and steel,imported iron and steel,traded iron and steel,2035,28.5150


In [369]:
i = 109
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df.head()

cement production by tech and vintage


,Units,scenario,region,sector,subsector,technology,output,Year,value
0,Mt,Current-Policy,South Korea,cement,cement,"cement CCS,year=2025",cement,2025,0.003206
1,Mt,Current-Policy,South Korea,cement,cement,"cement CCS,year=2025",cement,2030,0.002786
2,Mt,Current-Policy,South Korea,cement,cement,"cement CCS,year=2025",cement,2035,0.002090
3,Mt,Current-Policy,South Korea,cement,cement,"cement CCS,year=2030",cement,2030,0.010940
4,Mt,Current-Policy,South Korea,cement,cement,"cement CCS,year=2030",cement,2035,0.009514


In [370]:
df['tech'] = df['technology'].str.split(',').str[0]
dfFig = df[(df['Year'] >= 2005)].groupby(['scenario', 'Year', 'tech'])['value'].sum().reset_index()
dfFig.head()

,scenario,Year,tech,value
0,Current-Policy,2005,cement,53.82570
1,Current-Policy,2010,cement,52.50110
2,Current-Policy,2015,cement,52.50110
3,Current-Policy,2020,cement,46.29110
4,Current-Policy,2025,cement,41.96302


In [371]:
# stack_order = [
#     'BF', 'BF-CCS', 'BF-H2', 'BF-Biomass', 'EAF-scrap', 'DRI-EAF', 'DRI-EAF-CCS', 'DRI-EAF-H2', 
# ]
stack_order = [
    'cement', 'cement LC3', 'cement CCS'
]

In [372]:
df['tech'] = pd.Categorical(df['tech'], categories=stack_order, ordered=True)
df = df.sort_values(by=['Year', 'tech'])
df['tech'].unique()

['cement', 'cement LC3', 'cement CCS']
Categories (3, object): ['cement' < 'cement LC3' < 'cement CCS']

In [373]:
dfAgg1 = dfFig[(dfFig['scenario'] == 'Current-Policy')]
fig1 = px.bar(dfAgg1, x="Year", y="value", color="tech", title="Current Policy")
dfAgg2 = dfFig[(dfFig['scenario'] == 'Enhanced-Ambition')]
fig2 = px.bar(dfAgg2, x="Year", y="value", color="tech", title="Enhanced Ambition")

In [374]:
# ---- Colour & hatch (“pattern”) settings ------------------------
fuel_colors = {
    "cement LC3": "#FFB785",    # peach
    "cement" : "grey",   # vivid red
    "cement CCS"    : "#0057B5",   # deep blue
}

# optional: hatch / pattern overlay for the categories that are
# drawn with diagonal stripes in the figure
fuel_patterns = {

}

# Example of applying in Plotly
import plotly.graph_objects as go

def bar_for(fuel, x, y):
    return go.Bar(
        name   = fuel,
        x      = x,
        y      = y,
        marker = dict(
            color   = fuel_colors[fuel],
            pattern = dict(shape = fuel_patterns.get(fuel, ""))
        )
    )

In [375]:
years = list(range(2005, 2036, 5))

# Create subplots with secondary y-axes
fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    subplot_titles=("Current Policy", "Enhanced Ambition")
)
# === Legend Group: Gases ===

# ------------------------------------------------------------------
# add traces from the first figure → left pane
# ------------------------------------------------------------------
for tr in fig1.data:
    tr.showlegend = False                      # keep legend single
    # NEW → colour & pattern injection
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            # marker.pattern is available from Plotly 5.3+
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=1, secondary_y=False)

# ------------------------------------------------------------------
# add traces from the second figure → right pane
# ------------------------------------------------------------------
for tr in fig2.data:
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=2, secondary_y=False)


fig.update_layout(
    yaxis=dict(title="EJ", showgrid=True),
    yaxis1=dict(title="EJ", showgrid=True, title_font_size=20),
    
    # Hide secondary y-axis for col 1 (still used internally)
    yaxis2=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    
    # Show secondary y-axis for col 2
    yaxis4=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 60]
    ),
    barmode='stack',
    plot_bgcolor='rgba(0,0,0,0)',
    width=800, height=700,
)


fig.update_xaxes(tickangle=45)

fig.update_yaxes(range=[None, 60])

fig.update_layout(
    yaxis=dict(title="Mtoe", showgrid=True, gridcolor='lightgray'),
    yaxis3=dict(showgrid=True, gridcolor='lightgray'),
    title=dict(
        text="<b>Cement Production by Technology</b>",
        font=dict(size=28),
        x=0.5
    ),
    legend=dict(
        traceorder="reversed",
        font=dict(size=20),
        x=1.02, y=1,
        borderwidth=0
    )
)
fig.update_xaxes(
    tickvals=years,
    ticktext=[str(y) for y in years]
)

# adjust axis labels and ticks
fig.update_xaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_yaxes(title_font=dict(size=18), tickfont=dict(size=18))

# bump the subplot titles
fig.update_annotations(font=dict(size=21))

# (if you also set a main title earlier)
fig.update_layout(title=dict(font=dict(size=28)))
fig

In [376]:
i = 100
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df#.head()

industry final energy by tech and fuel


,Units,scenario,region,sector,subsector,technology,input,Year,value
0,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2025,0.000057
1,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2030,0.000278
2,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2035,0.000585
3,EJ,Current-Policy,South Korea,agricultural energy use,mobile,hydrogen,H2 wholesale dispensing,2025,0.000054
4,EJ,Current-Policy,South Korea,agricultural energy use,mobile,hydrogen,H2 wholesale dispensing,2030,0.000204
...,...,...,...,...,...,...,...,...,...
1245,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2015,0.002158
1246,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2020,0.002360
1247,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2025,0.002459
1248,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2030,0.003219


In [377]:
df['sector'].unique()

array(['agricultural energy use', 'ammonia', 'cement',
       'chemical energy use', 'chemical feedstocks',
       'construction energy use', 'construction feedstocks',
       'food processing', 'iron and steel', 'mining energy use',
       'other industrial energy use', 'other industrial feedstocks',
       'paper', 'process heat cement', 'process heat food processing',
       'process heat paper', 'waste biomass for paper', 'CO2 removal',
       'process heat dac'], dtype=object)

In [378]:
dfFig = df[(df['Year'] >= 2005) & (df['sector'] == 'chemical feedstocks')].copy()

In [379]:
dfFig['input'].unique()

array(['delivered coal', 'refined liquids industrial',
       'delivered biomass'], dtype=object)

In [380]:
def cat_fuel(fuel):

    if fuel in ['delivered coal']:
        return 'Coal'
    elif fuel == 'refined liquids industrial':
        return 'Oil'
    elif fuel in ['delivered biomass']:
        return 'Biomass'

In [381]:
dfFig['fuel'] = dfFig['input'].apply(cat_fuel)

In [382]:
stack_order = [
    'Oil', 'Biomass', 'Coal'
]

In [383]:
dfFig['fuel'] = pd.Categorical(dfFig['fuel'], categories=stack_order, ordered=True)
dfFig = dfFig.sort_values(by=['Year', 'fuel'])
dfFig['fuel'].unique()

['Oil', 'Coal', 'Biomass']
Categories (3, object): ['Oil' < 'Biomass' < 'Coal']

In [384]:
dfFig['value'] *= 23.8846

In [385]:
dfAgg1 = dfFig[(dfFig['scenario'] == 'Current-Policy')]
fig1 = px.bar(dfAgg1, x="Year", y="value", color="fuel", title="Current Policy")
dfAgg2 = dfFig[(dfFig['scenario'] == 'Enhanced-Ambition')]
fig2 = px.bar(dfAgg2, x="Year", y="value", color="fuel", title="Enhanced Ambition")

In [386]:
# ---- Colour & hatch (“pattern”) settings ------------------------
fuel_colors = {
    "Oil" : "#EF0C0C",   # vivid red
    "Coal"           : "#000000",   # solid black
    "Biomass"        : "#099B43",   # medium green
}

# optional: hatch / pattern overlay for the categories that are
# drawn with diagonal stripes in the figure
fuel_patterns = {

}

# Example of applying in Plotly
import plotly.graph_objects as go

def bar_for(fuel, x, y):
    return go.Bar(
        name   = fuel,
        x      = x,
        y      = y,
        marker = dict(
            color   = fuel_colors[fuel],
            pattern = dict(shape = fuel_patterns.get(fuel, ""))
        )
    )

In [387]:
years = list(range(2005, 2036, 5))

# Create subplots with secondary y-axes
fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    subplot_titles=("Current Policy", "Enhanced Ambition")
)
# === Legend Group: Gases ===

# ------------------------------------------------------------------
# add traces from the first figure → left pane
# ------------------------------------------------------------------
for tr in fig1.data:
    tr.showlegend = False                      # keep legend single
    # NEW → colour & pattern injection
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            # marker.pattern is available from Plotly 5.3+
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=1, secondary_y=False)

# ------------------------------------------------------------------
# add traces from the second figure → right pane
# ------------------------------------------------------------------
for tr in fig2.data:
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=2, secondary_y=False)


fig.update_layout(
    yaxis=dict(title="EJ", showgrid=True),
    yaxis1=dict(title="EJ", showgrid=True, title_font_size=20),
    
    # Hide secondary y-axis for col 1 (still used internally)
    yaxis2=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    
    # Show secondary y-axis for col 2
    yaxis4=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 60]
    ),
    barmode='stack',
    plot_bgcolor='rgba(0,0,0,0)',
    width=800, height=700,
)


fig.update_xaxes(tickangle=45)

fig.update_yaxes(range=[None, 60])

fig.update_layout(
    yaxis=dict(title="Mtoe", showgrid=True, gridcolor='lightgray'),
    yaxis3=dict(showgrid=True, gridcolor='lightgray'),
    title=dict(
        text="<b>Chemical Feedstock Inputs</b>",
        font=dict(size=28),
        x=0.5
    ),
    legend=dict(
        traceorder="reversed",
        font=dict(size=20),
        x=1.02, y=1,
        borderwidth=0
    )
)
fig.update_xaxes(
    tickvals=years,
    ticktext=[str(y) for y in years]
)

# adjust axis labels and ticks
fig.update_xaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_yaxes(title_font=dict(size=18), tickfont=dict(size=18))

# bump the subplot titles
fig.update_annotations(font=dict(size=21))

# (if you also set a main title earlier)
fig.update_layout(title=dict(font=dict(size=28)))
fig

In [418]:
i = 119
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

iron and steel production by tech


,Units,scenario,region,sector,subsector,technology,output,Year,value
0,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,1975,0.665514
1,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,1990,13.226000
2,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,2005,26.767200
3,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,2010,34.108100
4,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,2015,48.479100
...,...,...,...,...,...,...,...,...,...
73,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,EAF with scrap,iron and steel,2015,20.863000
74,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,EAF with scrap,iron and steel,2020,21.301980
75,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,EAF with scrap,iron and steel,2025,21.714970
76,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,EAF with scrap,iron and steel,2030,24.492600


In [419]:
df[(df['Year'] == 2025)]

,Units,scenario,region,sector,subsector,technology,output,Year,value
6,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,2025,44.444830
9,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,BLASTFUR CCS,iron and steel,2025,0.464729
12,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,BLASTFUR with hydrogen,iron and steel,2025,0.485242
15,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,Biomass-based,iron and steel,2025,0.610629
21,Mt,Current-Policy,South Korea,iron and steel,EAF with DRI,EAF with DRI,iron and steel,2025,0.452612
24,Mt,Current-Policy,South Korea,iron and steel,EAF with DRI,EAF with DRI CCS,iron and steel,2025,0.011474
27,Mt,Current-Policy,South Korea,iron and steel,EAF with DRI,Hydrogen-based DRI,iron and steel,2025,0.045943
36,Mt,Current-Policy,South Korea,iron and steel,EAF with scrap,EAF with scrap,iron and steel,2025,21.754880
45,Mt,Enhanced-Ambition,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,2025,44.379520
48,Mt,Enhanced-Ambition,South Korea,iron and steel,BLASTFUR,BLASTFUR CCS,iron and steel,2025,0.461438


In [420]:
df.groupby(['Year', 'scenario'])['value'].sum()

Year  scenario         
1975  Current-Policy        1.163048
      Enhanced-Ambition     1.163048
1990  Current-Policy       23.124960
      Enhanced-Ambition    23.124960
2005  Current-Policy       47.820000
      Enhanced-Ambition    47.820000
2010  Current-Policy       58.913960
      Enhanced-Ambition    58.913960
2015  Current-Policy       69.669959
      Enhanced-Ambition    69.669959
2020  Current-Policy       69.826103
      Enhanced-Ambition    69.826691
2025  Current-Policy       68.270339
      Enhanced-Ambition    68.206869
2030  Current-Policy       71.806983
      Enhanced-Ambition    64.471713
2035  Current-Policy       72.026203
      Enhanced-Ambition    63.901359
Name: value, dtype: float64

In [421]:
df['technology'].unique()

array(['BLASTFUR', 'BLASTFUR CCS', 'BLASTFUR with hydrogen',
       'Biomass-based', 'EAF with DRI', 'EAF with DRI CCS',
       'Hydrogen-based DRI', 'EAF with scrap'], dtype=object)

In [422]:
def cat_tech(tech):

    if tech == 'BLASTFUR':
        return 'BF'
    # if tech in [
    #     'BLASTFUR', 'BLASTFUR CCS', 'BLASTFUR with hydrogen', 'Biomass-based'
    # ]:
    #     return 'BF'
    elif tech == 'BLASTFUR CCS':
        return 'BF-CCS'
    elif tech == 'BLASTFUR with hydrogen':
        return 'BF-H2'
    elif tech == 'Biomass-based':
        return 'BF-Biomass'
    elif tech == 'EAF with DRI':
        return 'DRI-EAF'
    elif tech == 'EAF with DRI CCS':
        return 'DRI-EAF-CCS'
    elif tech == 'Hydrogen-based DRI':
        return 'DRI-EAF-H2'
    elif tech == 'EAF with scrap':
        return 'EAF-scrap'

In [423]:
df['tech'] = df['technology'].apply(cat_tech)

In [424]:
stack_order = [
    'BF', 'BF-CCS', 'BF-H2', 'BF-Biomass', 'EAF-scrap', 'DRI-EAF', 'DRI-EAF-CCS', 'DRI-EAF-H2', 
]
# stack_order = [
#     'BF', 'EAF-scrap', 'DRI-EAF', 'DRI-EAF-CCS', 'DRI-EAF-H2', 
# ]

In [425]:
df['tech'] = pd.Categorical(df['tech'], categories=stack_order, ordered=True)
df = df.sort_values(by=['Year', 'tech'])
df['tech'].unique()

['BF', 'EAF-scrap', 'DRI-EAF', 'BF-CCS', 'BF-H2', 'BF-Biomass', 'DRI-EAF-CCS', 'DRI-EAF-H2']
Categories (8, object): ['BF' < 'BF-CCS' < 'BF-H2' < 'BF-Biomass' < 'EAF-scrap' < 'DRI-EAF' < 'DRI-EAF-CCS' < 'DRI-EAF-H2']

In [426]:
df.groupby(['scenario', 'Year'])['value'].sum()

scenario           Year
Current-Policy     1975     1.163048
                   1990    23.124960
                   2005    47.820000
                   2010    58.913960
                   2015    69.669959
                   2020    69.826103
                   2025    68.270339
                   2030    71.806983
                   2035    72.026203
Enhanced-Ambition  1975     1.163048
                   1990    23.124960
                   2005    47.820000
                   2010    58.913960
                   2015    69.669959
                   2020    69.826691
                   2025    68.206869
                   2030    64.471713
                   2035    63.901359
Name: value, dtype: float64

In [427]:
df.groupby(['scenario', 'Year', 'tech'])['value'].sum().reset_index().to_excel("./output/ironsteel_prod.xlsx", index=False)

/tmp/ipykernel_15979/3538276874.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [428]:
dfAgg1 = df[(df['Year'] >= 2005) & (df['scenario'] == 'Current-Policy')].groupby(['Year', 'tech'])['value'].sum().reset_index()
fig1 = px.bar(dfAgg1, x="Year", y="value", color="tech", title="Current Policy")
dfAgg2 = df[(df['Year'] >= 2005) & (df['scenario'] == 'Enhanced-Ambition')].groupby(['Year', 'tech'])['value'].sum().reset_index()
fig2 = px.bar(dfAgg2, x="Year", y="value", color="tech", title="Enhanced Ambition")

/tmp/ipykernel_15979/2787647340.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_15979/2787647340.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [429]:
# ---- Colour & hatch (“pattern”) settings ------------------------
fuel_colors = {
    "BF"           : "#000000",   # solid black
    "BF-CCS"       : "grey",   # same black, but we’ll add a hatch
    "BF-H2"            : "#E4EAED",   # bright blue
    "EAF-scrap"        : "#492681",   # lighter cyan-blue
    "BF-Biomass"        : "#099B43",   # medium green
    "DRI-EAF"    : "#0057B5",   # deep blue
    "DRI-EAF-H2"       : "#FED30B",   # golden yellow
    "DRI-EAF-CCS"            : "#4DDADC",   # white base; will rely on hatch only
}

# optional: hatch / pattern overlay for the categories that are
# drawn with diagonal stripes in the figure
fuel_patterns = {
    "BF-CCS": "/",   # forward-slash hatch
    "DRI-EAF-CCS" : "/",   # same
    "DRI-EAF-H2"     : "\\",  # back-slash hatch
}

# Example of applying in Plotly
import plotly.graph_objects as go

def bar_for(fuel, x, y):
    return go.Bar(
        name   = fuel,
        x      = x,
        y      = y,
        marker = dict(
            color   = fuel_colors[fuel],
            pattern = dict(shape = fuel_patterns.get(fuel, ""))
        )
    )

In [430]:
# tell px.bar your exact category order,
# so that color‐groups (and legend entries) come out in stack_order
fig1 = px.bar(
    dfAgg1,
    x="Year", y="value", color="tech",
    category_orders={'tech': stack_order},
    title="Current Policy"
)
fig2 = px.bar(
    dfAgg2,
    x="Year", y="value", color="tech",
    category_orders={'tech': stack_order},
    title="Enhanced Ambition"
)

# stack the bars, and keep legend trace order = data order
for fig in (fig1, fig2):
    fig.update_layout(
        barmode='stack',
        legend_traceorder='normal',   # ← respect the data‐order
    )

In [431]:
years = list(range(2005, 2036, 5))

# Create subplots with secondary y-axes
fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    subplot_titles=("Current Policy", "Enhanced Ambition")
)
# === Legend Group: Gases ===

# ------------------------------------------------------------------
# add traces from the first figure → left pane
# ------------------------------------------------------------------
for tr in fig1.data:
    tr.showlegend = False                      # keep legend single
    # NEW → colour & pattern injection
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            # marker.pattern is available from Plotly 5.3+
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=1, secondary_y=False)

# ------------------------------------------------------------------
# add traces from the second figure → right pane
# ------------------------------------------------------------------
for tr in fig2.data:
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=2, secondary_y=False)


fig.update_layout(
    yaxis=dict(title="EJ", showgrid=True),
    yaxis1=dict(title="EJ", showgrid=True, title_font_size=20),
    
    # Hide secondary y-axis for col 1 (still used internally)
    yaxis2=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    
    # Show secondary y-axis for col 2
    yaxis4=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    barmode='stack',
    plot_bgcolor='rgba(0,0,0,0)',
    width=800, height=700,
)


fig.update_xaxes(tickangle=45)

fig.update_yaxes(range=[None, 90])

fig.update_layout(
    yaxis=dict(title="Mt", showgrid=True, gridcolor='lightgray'),
    yaxis3=dict(showgrid=True, gridcolor='lightgray'),
    title=dict(
        text="<b>Iron & Steel Production by Technology</b>",
        font=dict(size=33),
        x=0.5
    ),
    legend=dict(
        traceorder="reversed",
        font=dict(size=20),
        x=1.02, y=1,
        borderwidth=0
    )
)
fig.update_xaxes(
    tickvals=years,
    ticktext=[str(y) for y in years]
)

# adjust axis labels and ticks
fig.update_xaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_yaxes(title_font=dict(size=18), tickfont=dict(size=18))

# bump the subplot titles
fig.update_annotations(font=dict(size=21))

# (if you also set a main title earlier)
fig.update_layout(title=dict(font=dict(size=28)))

# fig.show()

fig

In [433]:
df[(df['Year'] >= 2015)].groupby(['scenario', 'Year', 'tech'])['value'].sum().reset_index().pivot(index=['scenario', 'Year'], columns=['tech'], values='value')

/tmp/ipykernel_15979/4170592175.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



tech                          BF     BF-CCS     BF-H2  BF-Biomass  EAF-scrap  \
scenario          Year                                                         
Current-Policy    2015  48.47910   0.000000  0.000000    0.000000  20.863000   
                  2020  48.16931   0.000000  0.000000    0.000000  21.301620   
                  2025  44.44483   0.464729  0.485242    0.610629  21.754880   
                  2030  42.26252   2.284012  1.475069    1.779638  22.349020   
                  2035  34.00245   5.085777  2.328811    2.894283  24.661310   
Enhanced-Ambition 2015  48.47910   0.000000  0.000000    0.000000  20.863000   
                  2020  48.16954   0.000000  0.000000    0.000000  21.301980   
                  2025  44.37952   0.461438  0.524449    0.608381  21.714970   
                  2030  23.87219   6.076209  4.050307    3.893751  24.492600   
                  2035   8.13800  11.210324  6.186806    5.658571  27.048707   

tech                     DRI-EAF  DRI-EAF-CCS  DRI-EAF-H2  
scenario          Year                                     
Current-Policy    2015  0.327859     0.000000    0.000000  
                  2020  0.355173     0.000000    0.000000  
                  2025  0.452612     0.011474    0.045943  
                  2030  1.293776     0.107650    0.255298  
                  2035  1.752835     0.191608    1.109129  
Enhanced-Ambition 2015  0.327859     0.000000    0.000000  
                  2020  0.355171     0.000000    0.000000  
                  2025  0.450257     0.011364    0.056491  
                  2030  0.987058     0.126108    0.973491  
                  2035  1.095155     0.195180    4.368616

In [434]:
df[(df['Year'] >= 2015)].groupby(['scenario', 'Year', 'tech'])['value'].sum().reset_index().pivot(index=['scenario', 'Year'], columns=['tech'], values='value').reset_index().to_excel("./output/ironsteel_production.xlsx", index=False)

/tmp/ipykernel_15979/2302511762.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [403]:
i = 100
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

industry final energy by tech and fuel


,Units,scenario,region,sector,subsector,technology,input,Year,value
0,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2025,0.000057
1,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2030,0.000278
2,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2035,0.000585
3,EJ,Current-Policy,South Korea,agricultural energy use,mobile,hydrogen,H2 wholesale dispensing,2025,0.000054
4,EJ,Current-Policy,South Korea,agricultural energy use,mobile,hydrogen,H2 wholesale dispensing,2030,0.000204
...,...,...,...,...,...,...,...,...,...
1245,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2015,0.002158
1246,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2020,0.002360
1247,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2025,0.002459
1248,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2030,0.003219


In [404]:
def cat_fuel(row):
    fuel = row['input']
    tech = row['technology']
    if tech in ['gas CCS', 'biomass CCS', 'coal CCS', 'refined liquids CCS']:
        return tech
    elif tech in ['hightemp DAC NG', 'hightemp DAC elec', 'lowtemp DAC heatpump']:
        return 'DAC'
    elif tech == 'electricity with solar':
        return 'electricity'
    elif tech == 'gas with solar':
        return 'gas' 
    elif fuel in ['H2 wholesale dispensing', 'H2 wholesale delivery', 'H2 industrial']:
        return 'hydrogen'
    elif fuel == 'elect_td_ind':
        return 'electricity'
    elif fuel in ['refined liquids industrial']:
        return 'refined liquids'
    elif fuel in ['delivered biomass', 'regional woodpulp for energy']:
        return 'biomass'
    elif fuel in ['wholesale gas']:
        return 'gas'
    elif fuel in ['delivered coal']:
        return 'coal'
    else:
        print(fuel, tech)
        return 'others'

In [405]:
df['Units'].unique()

array(['EJ'], dtype=object)

In [406]:
df['input'].unique()

array(['elect_td_ind', 'H2 wholesale dispensing',
       'refined liquids industrial', 'delivered biomass', 'wholesale gas',
       'H2 wholesale delivery', 'H2 industrial', 'delivered coal',
       'global solar resource', 'regional woodpulp for energy'],
      dtype=object)

In [407]:
df['fuel'] = df.apply(cat_fuel, axis=1)
df

,Units,scenario,region,sector,subsector,technology,input,Year,value,fuel
0,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2025,0.000057,electricity
1,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2030,0.000278,electricity
2,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2035,0.000585,electricity
3,EJ,Current-Policy,South Korea,agricultural energy use,mobile,hydrogen,H2 wholesale dispensing,2025,0.000054,hydrogen
4,EJ,Current-Policy,South Korea,agricultural energy use,mobile,hydrogen,H2 wholesale dispensing,2030,0.000204,hydrogen
...,...,...,...,...,...,...,...,...,...,...
1245,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2015,0.002158,biomass
1246,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2020,0.002360,biomass
1247,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2025,0.002459,biomass
1248,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2030,0.003219,biomass


In [408]:
df[(df['fuel'] == 'biomass') & (df['Year'] >= 2025) & (df['scenario'] == 'Enhanced-Ambition')].pivot(index=['sector', 'subsector', 'technology', 'input'], columns='Year', values='value')

Year                                                                                    2025  \
sector                       subsector  technology    input                                    
agricultural energy use      stationary biomass       delivered biomass             0.003017   
chemical energy use          biomass    biomass       delivered biomass             0.011618   
chemical feedstocks          biomass    biomass       delivered biomass                  NaN   
construction energy use      stationary biomass       delivered biomass             0.001197   
iron and steel               BLASTFUR   Biomass-based delivered biomass             0.010207   
other industrial energy use  biomass    biomass       delivered biomass             0.089550   
                                        biomass cogen delivered biomass             0.002641   
process heat cement          biomass    biomass       delivered biomass             0.037907   
process heat food processing biomass    biomass       delivered biomass             0.003870   
                                        biomass cogen delivered biomass             0.000002   
waste biomass for paper      biomass    biomass       regional woodpulp for energy  0.025230   
                                        biomass cogen regional woodpulp for energy  0.002459   

Year                                                                                    2030  \
sector                       subsector  technology    input                                    
agricultural energy use      stationary biomass       delivered biomass             0.001776   
chemical energy use          biomass    biomass       delivered biomass             0.011413   
chemical feedstocks          biomass    biomass       delivered biomass             0.102701   
construction energy use      stationary biomass       delivered biomass             0.001265   
iron and steel               BLASTFUR   Biomass-based delivered biomass             0.065329   
other industrial energy use  biomass    biomass       delivered biomass             0.087106   
                                        biomass cogen delivered biomass             0.002628   
process heat cement          biomass    biomass       delivered biomass             0.035837   
process heat food processing biomass    biomass       delivered biomass             0.003828   
                                        biomass cogen delivered biomass             0.000011   
waste biomass for paper      biomass    biomass       regional woodpulp for energy  0.024176   
                                        biomass cogen regional woodpulp for energy  0.003219   

Year                                                                                    2035  
sector                       subsector  technology    input                                   
agricultural energy use      stationary biomass       delivered biomass             0.000723  
chemical energy use          biomass    biomass       delivered biomass             0.010869  
chemical feedstocks          biomass    biomass       delivered biomass             0.158341  
construction energy use      stationary biomass       delivered biomass             0.001130  
iron and steel               BLASTFUR   Biomass-based delivered biomass             0.094939  
other industrial energy use  biomass    biomass       delivered biomass             0.084098  
                                        biomass cogen delivered biomass             0.002923  
process heat cement          biomass    biomass       delivered biomass             0.036232  
process heat food processing biomass    biomass       delivered biomass             0.003251  
                                        biomass cogen delivered biomass             0.000037  
waste biomass for paper      biomass    biomass       regional woodpulp for energy  0.021695  
                                        biomass cogen regional woodpulp for

In [409]:
df[(df['Year'] >= 2025) & (df['scenario'] == 'Enhanced-Ambition') & (df['sector'] == 'chemical feedstocks')]

,Units,scenario,region,sector,subsector,technology,input,Year,value,fuel
764,EJ,Enhanced-Ambition,South Korea,chemical feedstocks,biomass,biomass,delivered biomass,2030,0.102701,biomass
765,EJ,Enhanced-Ambition,South Korea,chemical feedstocks,biomass,biomass,delivered biomass,2035,0.158341,biomass
769,EJ,Enhanced-Ambition,South Korea,chemical feedstocks,coal,coal,delivered coal,2025,0.015175,coal
770,EJ,Enhanced-Ambition,South Korea,chemical feedstocks,coal,coal,delivered coal,2030,0.009336,coal
771,EJ,Enhanced-Ambition,South Korea,chemical feedstocks,coal,coal,delivered coal,2035,0.007219,coal
778,EJ,Enhanced-Ambition,South Korea,chemical feedstocks,refined liquids,refined liquids,refined liquids industrial,2025,1.899820,refined liquids
779,EJ,Enhanced-Ambition,South Korea,chemical feedstocks,refined liquids,refined liquids,refined liquids industrial,2030,1.711730,refined liquids
780,EJ,Enhanced-Ambition,South Korea,chemical feedstocks,refined liquids,refined liquids,refined liquids industrial,2035,1.527780,refined liquids


In [410]:
dfFig = df[(df['Year'] >= 2005)].groupby(['scenario', 'Year', 'fuel'])['value'].sum().reset_index()
dfFig

,scenario,Year,fuel,value
0,Current-Policy,2005,biomass,0.107670
1,Current-Policy,2005,coal,0.823345
2,Current-Policy,2005,electricity,0.671997
3,Current-Policy,2005,gas,0.213206
4,Current-Policy,2005,refined liquids,1.797392
...,...,...,...,...
108,Enhanced-Ambition,2035,gas,0.541161
109,Enhanced-Ambition,2035,gas CCS,0.044293
110,Enhanced-Ambition,2035,hydrogen,0.124605
111,Enhanced-Ambition,2035,refined liquids,1.927861


In [411]:
dfFig['value'] *= 23.8846

In [412]:
# dfFig['fuel'] = dfFig['fuel'].apply(lambda fuel: fuel.capitalize())
dfFig['fuel'].unique()

array(['biomass', 'coal', 'electricity', 'gas', 'refined liquids',
       'biomass CCS', 'coal CCS', 'gas CCS', 'hydrogen',
       'refined liquids CCS', 'DAC'], dtype=object)

In [413]:
stack_order = [
    'DAC', 'hydrogen', 'electricity', 'biomass CCS', 'biomass', 'gas CCS', 'gas', 'coal CCS', 'coal', 'refined liquids CCS', 'refined liquids'
]

In [414]:
dfFig['fuel'] = pd.Categorical(dfFig['fuel'], categories=stack_order, ordered=True)
dfFig = dfFig.sort_values(by=['Year', 'fuel'])
dfFig['fuel'].unique()

['electricity', 'biomass', 'gas', 'coal', 'refined liquids', ..., 'biomass CCS', 'gas CCS', 'coal CCS', 'refined liquids CCS', 'DAC']
Length: 11
Categories (11, object): ['DAC' < 'hydrogen' < 'electricity' < 'biomass CCS' ... 'coal CCS' < 'coal' < 'refined liquids CCS' < 'refined liquids']

In [415]:
dfAgg1 = dfFig[(dfFig['scenario'] == 'Current-Policy')]
fig1 = px.bar(dfAgg1, x="Year", y="value", color="fuel", title="Current Policy")
dfAgg2 = dfFig[(dfFig['scenario'] == 'Enhanced-Ambition')]
fig2 = px.bar(dfAgg2, x="Year", y="value", color="fuel", title="Enhanced Ambition")

In [416]:
# ---- Colour & hatch (“pattern”) settings ------------------------
fuel_colors = {
    "refined liquids CCS": "#FFB785",    # peach
    "refined liquids" : "#EF0C0C",   # vivid red
    "coal"           : "#000000",   # solid black
    "coal CCS"       : "#000000",   # same black, but we’ll add a hatch
    "gas"            : "#0186E0",   # bright blue
    "gas CCS"        : "#33A8E2",   # lighter cyan-blue
    "biomass"        : "#099B43",   # medium green
    "electricity"    : "#0057B5",   # deep blue
    "hydrogen"       : "#FED30B",   # golden yellow
    "DAC"            : "#FFFFFF",   # white base; will rely on hatch only
}

# optional: hatch / pattern overlay for the categories that are
# drawn with diagonal stripes in the figure
fuel_patterns = {
    "coal CCS": "/",   # forward-slash hatch
    "gas CCS" : "/",   # same
    "refined liquids CCS": "/",
    "biomass CCS": "/",
    "DAC"     : "\\",  # back-slash hatch
}

# Example of applying in Plotly
import plotly.graph_objects as go

def bar_for(fuel, x, y):
    return go.Bar(
        name   = fuel,
        x      = x,
        y      = y,
        marker = dict(
            color   = fuel_colors[fuel],
            pattern = dict(shape = fuel_patterns.get(fuel, ""))
        )
    )

In [417]:
years = list(range(2005, 2036, 5))

# Create subplots with secondary y-axes
fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    subplot_titles=("Current Policy", "Enhanced Ambition")
)
# === Legend Group: Gases ===

# ------------------------------------------------------------------
# add traces from the first figure → left pane
# ------------------------------------------------------------------
for tr in fig1.data:
    tr.showlegend = False                      # keep legend single
    # NEW → colour & pattern injection
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            # marker.pattern is available from Plotly 5.3+
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=1, secondary_y=False)

# ------------------------------------------------------------------
# add traces from the second figure → right pane
# ------------------------------------------------------------------
for tr in fig2.data:
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=2, secondary_y=False)


fig.update_layout(
    yaxis=dict(title="EJ", showgrid=True),
    yaxis1=dict(title="EJ", showgrid=True, title_font_size=20),
    
    # Hide secondary y-axis for col 1 (still used internally)
    yaxis2=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    
    # Show secondary y-axis for col 2
    yaxis4=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    barmode='stack',
    plot_bgcolor='rgba(0,0,0,0)',
    width=800, height=700,
)


fig.update_xaxes(tickangle=45)

fig.update_yaxes(range=[None, 150])

fig.update_layout(
    yaxis=dict(title="Mtoe", showgrid=True, gridcolor='lightgray'),
    yaxis3=dict(showgrid=True, gridcolor='lightgray'),
    title=dict(
        text="<b>Final energy consumption by fuel</b>",
        font=dict(size=28),
        x=0.5
    ),
    legend=dict(
        traceorder="reversed",
        font=dict(size=20),
        x=1.02, y=1,
        borderwidth=0
    )
)
fig.update_xaxes(
    tickvals=years,
    ticktext=[str(y) for y in years]
)

# adjust axis labels and ticks
fig.update_xaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_yaxes(title_font=dict(size=18), tickfont=dict(size=18))

# bump the subplot titles
fig.update_annotations(font=dict(size=21))

# (if you also set a main title earlier)
fig.update_layout(title=dict(font=dict(size=28)))
fig